# Azure AI Content Understanding — Underwriting Scenario Limitations

Field notes from building an SOV (Statement of Values) extraction pipeline
for commercial property submissions using the GA Content Understanding SDK.

**Scenario.** Carrier underwriting intake: brokers email submissions containing
an account-level cover document plus a per-location SOV. Inputs arrive in three
shapes:

| Mode | Source | Analyzer method |
|---|---|---|
| A | PDF SOV | `extract` (prebuilt-document) |
| B | `.xlsx` SOV | `generate` |
| C | `.xlsx` SOV + embedded images of overflow locations | `generate` × N (client-side fan-out) |

This notebook catalogs the limitations we hit and the mitigations we shipped.
All examples reference the working demo in [`demo/sov/`](../../demo/sov/).

---
## 1. File-format constraints

### 1.1 `.xlsx` is not supported by `method=extract`

Spreadsheets cannot flow through the OCR / layout pipeline that `extract` depends on.
They must use `method=generate` (LLM-driven), which has different trade-offs:
- No OCR grounding
- No bounding boxes / spans
- No per-field confidence scores
- Looser fidelity (model paraphrases, infers, occasionally fabricates)

#### What "not supported" actually looks like
The API does **not** raise an error. A `prebuilt-document` + custom field schema +
`method=extract` call against an `.xlsx` returns HTTP 200 with a fully-shaped JSON
response, and **every field has `confidence: 1`** — but the per-field `valueX`
keys are missing for almost everything. On the Acme SOV (19 locations, 18 top-level
fields, 22 location subfields) we measured:

| Bucket | Populated |
|---|---|
| Top-level scalars | **1 / 17** (only `Currency: USD`) |
| Locations array cardinality | 19 / 19 ✓ (rows detected correctly) |
| Locations cells (rows × subfields) | **0 / 418** |

The `confidence: 1` on the empty fields is misleading — it's confidence in the
*schema decision that the field has no grounded value*, not confidence in a value.
A naive consumer that trusts the response shape will believe the call succeeded.
Reproduction notebook: [`research_xlsx_extract.ipynb`](research_xlsx_extract.ipynb).

#### Why this happens (documented behavior, just not in one place)
Three independent doc statements combine to produce this outcome:

1. **Office files only get "Minimal" processing.**
   [Service quotas and limits → Input file limits](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/service-limits#input-file-limits) splits formats into
   processing tiers: `pdf, tiff, jpg, png, bmp, heif, heic` get **Basic (OCR) or
   Standard (Layout)**; `docx, xlsx, pptx` get **Minimal**. Only the first tier
   produces the layout/OCR spans that `extract` needs.

2. **`extract` is a grounded extraction method.**
   [Document overview → Field extraction methods](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/document/overview#field-extraction-methods) describes `extract` as
   "precise and focused information capture," paired with the
   `estimateFieldSourceAndConfidence` config — both of which require OCR/layout
   spans to ground against. `generate` is the LLM-over-markdown path with no
   layout dependency.

3. **Only four analyzers can be used as `baseAnalyzerId`.**
   [Prebuilt analyzers → Base analyzers](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/concepts/prebuilt-analyzers#base-analyzers): "Currently, you can only derive
   custom analyzers from this set of four base analyzers: `prebuilt-audio`,
   `prebuilt-document`, `prebuilt-image`, `prebuilt-video`." So the obvious
   workaround — swap the base to `prebuilt-layout` or `prebuilt-documentSearch` —
   is rejected with `InvalidBaseAnalyzerId`. Confirmed empirically (variants 4–6
   in the research notebook).

The PDF-only-rich-features pattern is consistent across the docs (e.g.
`prebuilt-layout`: "Detects figure types ... PDF files only"; `prebuilt-documentSearch`:
"Figure analysis is only supported for PDF and image file formats").

#### Practical consequence
For xlsx, `method=extract` is **structurally not viable**, not a configuration
bug. There is no analyzer-side workaround in the current API. The intended path
is `method=generate` over the layout-derived markdown, which is what
`sovGenerateV1` (Patterns B and C) does. We added a `Recommended for: PDF` hint
to Pattern A in the workshop UI to keep attendees from trusting the misleading
all-`confidence: 1` xlsx response.

### 1.2 No native multi-modal call
An `.xlsx` workbook with embedded/attached images cannot be processed as one
document. We implemented **Pattern C**: one `generate` call for the workbook +
one `generate` call per extracted image, then merged results client-side.

### 1.3 Image extraction from xlsx is a client-side problem
The service does not unpack workbook images. We extracted them with
`openpyxl` / `zipfile` before fan-out.


In [ ]:
# Pattern C — client-side fan-out + merge (sketch)
# Full implementation: demo/sov/notebooks/01_extract_sov.ipynb
from pathlib import Path
import zipfile

def extract_xlsx_images(xlsx_path: Path, out_dir: Path) -> list[Path]:
    """xlsx is a zip; embedded images live under xl/media/."""
    out_dir.mkdir(parents=True, exist_ok=True)
    images = []
    with zipfile.ZipFile(xlsx_path) as z:
        for name in z.namelist():
            if name.startswith("xl/media/") and name.lower().endswith((".png", ".jpg", ".jpeg")):
                target = out_dir / Path(name).name
                target.write_bytes(z.read(name))
                images.append(target)
    return images

# Then: one generate call per file, merge by (LocationNumber, lower(Street)),
# field-level complement, recompute LocationCount = len(merged_locations).

---
## 2. Response-length / output-size limits

### 2.1 `method=generate` silently truncates long arrays
Large `Locations[]` arrays (20+ rows × many fields) get cut off or fields
collapse to `null`. We hit this on the Acme submission (22 locations) and had
to keep the schema lean and lean on Pattern C merging.

### 2.2 String fields truncate mid-sentence
Long free-text fields (e.g. `primary_operations`) come back ending in `…` or
mid-word. We hardened the validator to treat `prefix…` ≈ full string.

### 2.3 No documented hard token cap
The limit is implicit — you discover it by output corruption, not by an error
code or warning.

In [ ]:
# Validator normalization to absorb truncation (excerpt)
# Full impl: demo/sov/notebooks/02_validate_extraction.ipynb
import re

def _norm(v):
    if v is None:
        return None
    s = str(v).strip()
    # Strip trailing ellipsis / em-dash truncation markers
    s = re.sub(r"[\u2026\.]{1,3}$", "", s).strip()
    # Footnote markers like "ABC Corp (1)"
    s = re.sub(r"\s*\(\d+\)\s*$", "", s)
    # MARGIN NOTE: prefix
    s = re.sub(r"^MARGIN NOTE:\s*", "", s, flags=re.IGNORECASE)
    return s.casefold()

def matches(expected: str, actual: str) -> bool:
    e, a = _norm(expected), _norm(actual)
    if e == a:
        return True
    # Allow truncation: actual is a prefix of expected
    return bool(e and a and e.startswith(a))

---
## 3. Cell-level vs. text-level extraction (xlsx `generate`)

The most surprising class of issues. The model often treats the workbook as a
grid of cells rather than as a structured document.

### 3.1 Raw cell value instead of semantic text
Instead of returning `"123 Main St, Suite 400, Dallas, TX 75201"` assembled
from adjacent cells, it returns the contents of one cell — losing
concatenation across street + suite or city + state + zip. Sometimes a header
label is returned as if it were a value.

### 3.2 No awareness of merged cells
Headers spanning `A:D` either land under the wrong field or get duplicated
into every row beneath them.

### 3.3 Column-vs-row orientation confusion
Workbooks shaped as `A=key, B=value` (vertical) sometimes get read as if A
were a header row, swapping field names with values.

### 3.4 Numeric formatting leaks
Currency cells return as `"$1,250,000"` (string) in some rows and `1250000`
(number) in others within the same call. Type stability is per-row, not
per-field.

### 3.5 Date cells return as Excel serial numbers
Number-formatted date cells come back as `45292` rather than `2024-01-01`.

### 3.6 No structural grounding
Unlike `extract` (spans + bounding boxes), `generate` on xlsx gives you no way
to trace a value back to a cell address (`B7`). You cannot verify or correct
cell-level mistakes programmatically.

### 3.7 In-cell whitespace/newlines flatten
Multi-line cell content (common in `Notes` / `Operations` columns) collapses
to a single line, dropping list structure.

In [ ]:
# Illustrative payload shapes we observed (anonymized).
# These are representative of what came back from method=generate on xlsx,
# NOT live calls.
examples = {
    "3.1 raw cell vs. semantic text": {
        "expected": "123 Main St, Suite 400, Dallas, TX 75201",
        "observed": "123 Main St",   # Suite + city/state/zip dropped
    },
    "3.2 merged-cell header leak": {
        "expected_field_value": "Owner-occupied",
        "observed_field_value": "PROPERTY DETAILS",  # merged header bled into rows
    },
    "3.3 row/column flip": {
        "expected": {"insured_name": "Acme Logistics", "effective_date": "2025-06-01"},
        "observed": {"Acme Logistics": "insured_name", "2025-06-01": "effective_date"},
    },
    "3.4 mixed numeric types": {
        "locations[0].tiv": "$1,250,000",
        "locations[1].tiv": 1250000,
        "locations[2].tiv": None,
    },
    "3.5 excel serial date": {
        "expected": "2024-01-01",
        "observed": 45292,
    },
    "3.7 newline flatten": {
        "expected": "Sprinklered\nCentral station alarm\n24/7 security",
        "observed": "Sprinklered Central station alarm 24/7 security",
    },
}

for k, v in examples.items():
    print(f"\n[{k}]")
    print(v)

---
## 4. Mitigations applied in this repo

| Limitation | Mitigation |
|---|---|
| 1.1 No xlsx in `extract` | Use `generate`; UI hint that Pattern A is PDF-recommended |
| 1.2 No multi-modal call | **Pattern C**: client-side fan-out + field-level merge |
| 1.3 Image unpacking | `zipfile` over `xl/media/` |
| 1.4 Row collisions in vector PDFs | Rasterize the PDF to 600 DPI multi-page TIFF before sending — forces CU's OCR path |
| 2.1 Array truncation | Lean schema; merge step recovers missed rows; recompute `LocationCount` post-merge |
| 2.2 String truncation | Validator treats `prefix…` ≈ full string |
| 3.1 Raw cell values | Tighten field `description` ("return the full assembled address …") |
| 3.2 Merged-cell leaks | Validator dedupes / drops header-shaped values |
| 3.4–3.5 Type/date drift | Client-side coercion in post-processing |
| 3.6 No grounding | Cache outputs; treat `generate` as best-effort, validate against ground truth |

## 5. Asks for the service

1. **Fail loudly when `extract` is called on a Minimal-tier format.** Return a
   `400 InvalidRequest` (or at minimum `_meta.warnings`) instead of HTTP 200 with
   `confidence: 1` on every empty field. The current behavior is silently
   misleading — the response shape implies success.
2. **Surface the file-format → processing-tier mapping at analyzer-create time.**
   When a custom analyzer with `method=extract` fields is created, validate at
   creation that the configured `baseAnalyzerId` produces grounding for the
   formats the customer expects to send.
3. **Native multi-modal analyzer call** for `xlsx + images` (eliminate Pattern C).
4. **Explicit error on output truncation** instead of silent corruption; expose
   the token/array-length budget.
5. **Cell-address grounding** in `generate` responses for xlsx (return
   `{value, sourceRef: "Sheet1!B7"}`).
6. **Strict mode / no-hallucination flag** that forces `null` when a field
   isn't grounded in the source.
7. **Determinism controls** (`seed`, `temperature=0`) on `generate`.
8. **Per-field confidence scores** for `generate`, parallel to `extract`.
9. **`forceOcr` (or equivalent) flag on `prebuilt-layout`** for borderless
   tables in vector PDFs. Today the layout analyzer reads the embedded text
   stream and merges adjacent rows when no cell borders separate them — and
   the merge sometimes inverts the row order. Forcing OCR fixes it but
   currently requires client-side rasterization, which inflates payload
   ~17× and adds ~2s of latency. **Full evidence + repro:**
   [`research-output/pdfs/BUG_REPORT.md`](research-output/pdfs/BUG_REPORT.md).
10. **Spatial sort before row clustering.** As a partial fix even without
    `forceOcr`, sort text-stream emissions by `(y, x)` of bounding-box
    centers before grouping. This would at least eliminate the *order
    inversion* even when rows still get merged.
11. **Structured warning when row segmentation is suspect.** Today the only
    signal that something went wrong with row segmentation is a `<br>`
    separator inside a `<td>` in the markdown. A `_meta.warnings: [{ code:
    "PossibleRowMerge", page: N, ... }]` entry would let downstream
    pipelines detect and triage automatically.

## 6. Reproducibility

- Empirical reproduction of limitation 1.1 (with raw payloads for all 7
  hypothesis variants): [`research_xlsx_extract.ipynb`](research_xlsx_extract.ipynb).
- Empirical reproduction of limitation 1.4 (vector PDF row collisions vs.
  TIFF OCR fix): [`research_xlsx_to_pdf.ipynb`](research_xlsx_to_pdf.ipynb)
  and [`research-output/pdfs/BUG_REPORT.md`](research-output/pdfs/BUG_REPORT.md).
- Working SOV pipeline (Patterns A/B/C end to end):
  [`demo/sov/`](../../demo/sov/).
- Workshop UI exposing the same patterns interactively: [`apps/workshop/`](../../apps/workshop/).
